# pixel-trust v5: learned spatial trust head

A small U-Net over the nine trust-signal maps replaces the per-pixel tree combiner. No generation: it reads the per-pixel features saved by earlier runs, so add three inputs (notebook outputs): **pixel-trust v2 SOTA comparison** (KITTI, SD inpainting), **pixel-trust v3 GenWarp** (KITTI, GenWarp) and **pixel-trust v4 Middlebury transfer** (Middlebury, both generators).

Compared with identical conformal calibration (alpha = 0.1): trees (v1-v4 ours), published signals, trust head (9 signals), trust head (9 + published), and one generator-aware head trained jointly on both generators. On KITTI, on Middlebury (200 random scene-level splits) and for KITTI -> Middlebury transfer.

Settings: Accelerator **GPU T4 x2** (one is enough), Internet **on**.

In [ ]:
import os, sys, glob, json, subprocess
REPO = "https://github.com/Rahul-Verma97/pixel-trust.git"
if not os.path.isdir("/kaggle/working/pixel-trust"):
    !git clone -q {REPO} /kaggle/working/pixel-trust
os.chdir("/kaggle/working/pixel-trust"); sys.path.insert(0, "/kaggle/working/pixel-trust")
!git log -1 --format='%h %s'
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
# find every saved feature directory and label it by dataset and generator
import numpy as np
found = {}
for d in sorted({os.path.dirname(os.path.dirname(f)) for f in glob.glob("/kaggle/input/**/feats/*_clean.npz", recursive=True)}):
    files = glob.glob(os.path.join(d, "feats", "*.npz"))
    z = np.load(files[0])
    ds = str(z["dataset"]) if "dataset" in z.files else "kitti"
    logs = glob.glob(os.path.join(d, "log_shard*.jsonl"))
    model = json.loads(open(logs[0]).readline()).get("model", "?") if logs else "?"
    gen = "genwarp" if "genwarp" in model.lower() else "sd"
    print(f"{ds:10s} {gen:8s} {len(files):4d} files  {d}")
    key = f"{ds}_{gen}"
    if key not in found or len(files) > found[key][1]:
        found[key] = (d, len(files))
print({k: v[1] for k, v in found.items()})
need = ["kitti_sd", "kitti_genwarp", "middlebury_sd", "middlebury_genwarp"]
assert all(k in found for k in need), f"missing inputs: {[k for k in need if k not in found]}"

In [ ]:
OUT = "/kaggle/working/head"
args = f'--kitti-sd {found["kitti_sd"][0]} --kitti-gw {found["kitti_genwarp"][0]} --mb-sd {found["middlebury_sd"][0]} --mb-gw {found["middlebury_genwarp"][0]} --out {OUT}'
!python run_head.py {args} 2>&1 | grep -v Warning

In [ ]:
from IPython.display import Image, display
print(open(f"{OUT}/REPORT_head.md").read())
display(Image(f"{OUT}/fig_head.png"))

In [ ]:
!cd /kaggle/working && zip -q -r pixel_trust_v5_head.zip head && ls -la pixel_trust_v5_head.zip